# Train ST-GCN++ for two badminton actions

Classes: `backhand_drive` and `forehand_lift`. This model has no `other` or `unknown` output. Run the cells in order on a T4 GPU.

In [ ]:
!nvidia-smi
!git clone --recurse-submodules https://github.com/dattt-cy/Badminton_AI.git /content/Badminton_AI
%cd /content/Badminton_AI

## Upload the two-class annotation file

Choose `badminton_actions_2class.pkl` from your computer.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
source = Path(next(iter(uploaded)))
destination = Path('/content/Badminton_AI/data/annotations/badminton_actions.pkl')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source), destination)
print(destination, destination.stat().st_size)

## Install Conda once

This restarts the runtime. Reconnect and continue with the following cell; do not run this cell again.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%cd /content/Badminton_AI
!CONDA_SOLVER=classic conda create -y -n pyskl_310 python=3.10 pip
!conda run -n pyskl_310 python -m pip install --no-cache-dir setuptools==69.5.1 numpy==1.23.5 scipy==1.9.3 pyyaml tqdm addict yapf==0.32.0 packaging termcolor pillow opencv-python-headless==4.7.0.72 fvcore==0.1.5.post20221221
!conda run -n pyskl_310 python -m pip install --no-cache-dir torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!conda run -n pyskl_310 python -m pip install --no-cache-dir --no-deps mmcv-full==1.7.0 -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.12.0/index.html
!conda run -n pyskl_310 python -m pip install --no-build-isolation --no-deps -e external/pyskl
!conda run -n pyskl_310 python -m pip install --no-deps -e .
!conda run -n pyskl_310 python -m pip uninstall -y opencv-python
!conda run -n pyskl_310 python -m pip install --no-cache-dir --force-reinstall numpy==1.23.5 opencv-python-headless==4.7.0.72

In [ ]:
!conda run -n pyskl_310 python -c "import torch, mmcv, pyskl; print('torch', torch.__version__, 'mmcv', mmcv.__version__, 'cuda', torch.cuda.is_available())"

## Train, validate, and test the best checkpoint

In [ ]:
!PYTHONUNBUFFERED=1 conda run --no-capture-output -n pyskl_310 python -u -m torch.distributed.run --nproc_per_node=1 scripts/training/run_pyskl_numpy_compat.py external/pyskl/tools/train.py configs/action_recognition/stgcnpp_badminton.py --launcher pytorch --validate --test-best

In [ ]:
from google.colab import files
!zip -qr /content/stgcnpp_badminton_2class_results.zip work_dirs/stgcnpp_badminton
files.download('/content/stgcnpp_badminton_2class_results.zip')